# RAG 3: bardziej złożony pipeline

Ten przykład pokazuje bardziej rozbudowany system:
- **chunking z metadanymi**,
- **hybrydowe wyszukiwanie**: TF-IDF + dopasowanie słów kluczowych,
- **reranking**,
- **pamięć konwersacji**,
- odpowiedź z **cytowaniem źródeł**.

To nadal działa offline, ale architektura przypomina już prawdziwy mini-system RAG.

In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from collections import Counter
import pandas as pd
import re

In [2]:
knowledge_base = [
    {
        "doc_id": "doc_transformers",
        "section": "attention",
        "text": "Mechanizm self-attention pozwala modelowi ocenić znaczenie różnych słów w kontekście całego zdania."
    },
    {
        "doc_id": "doc_transformers",
        "section": "applications",
        "text": "Transformery są używane w tłumaczeniu, streszczaniu, systemach dialogowych oraz wyszukiwaniu semantycznym."
    },
    {
        "doc_id": "doc_rag",
        "section": "definition",
        "text": "RAG to architektura, w której model generujący korzysta z zewnętrznie pobranych fragmentów wiedzy."
    },
    {
        "doc_id": "doc_rag",
        "section": "benefits",
        "text": "Zaletą RAG jest redukcja halucynacji, większa kontrola nad źródłami i możliwość aktualizacji wiedzy bez pełnego trenowania modelu."
    },
    {
        "doc_id": "doc_rag",
        "section": "limitations",
        "text": "Ograniczenia RAG obejmują słaby retrieval, błędny chunking, niską jakość danych oraz problemy z rerankingiem."
    },
    {
        "doc_id": "doc_embeddings",
        "section": "meaning",
        "text": "Embedding to wektorowa reprezentacja tekstu, która pozwala mierzyć podobieństwo semantyczne między fragmentami."
    },
    {
        "doc_id": "doc_embeddings",
        "section": "usage",
        "text": "Embeddingi są szeroko stosowane w wyszukiwaniu semantycznym, rekomendacjach i klastrowaniu dokumentów."
    }
]

kb_df = pd.DataFrame(knowledge_base)
kb_df

,doc_id,section,text
0,doc_transformers,attention,Mechanizm self-attention pozwala modelowi ocen...
1,doc_transformers,applications,"Transformery są używane w tłumaczeniu, streszc..."
2,doc_rag,definition,"RAG to architektura, w której model generujący..."
3,doc_rag,benefits,"Zaletą RAG jest redukcja halucynacji, większa ..."
4,doc_rag,limitations,"Ograniczenia RAG obejmują słaby retrieval, błę..."
5,doc_embeddings,meaning,"Embedding to wektorowa reprezentacja tekstu, k..."
6,doc_embeddings,usage,Embeddingi są szeroko stosowane w wyszukiwaniu...


In [3]:
vectorizer = TfidfVectorizer(stop_words=None)
kb_matrix = vectorizer.fit_transform(kb_df["text"])

def tokenize(text):
    return re.findall(r"\w+", text.lower())

def keyword_overlap_score(query, text):
    q = set(tokenize(query))
    t = set(tokenize(text))
    if not q:
        return 0.0
    return len(q & t) / len(q)

def hybrid_retrieve(query, top_k=5, alpha=0.7):
    q_vec = vectorizer.transform([query])
    semantic_scores = cosine_similarity(q_vec, kb_matrix).flatten()

    results = []
    for i, row in kb_df.iterrows():
        kw_score = keyword_overlap_score(query, row["text"])
        final_score = alpha * float(semantic_scores[i]) + (1 - alpha) * kw_score
        results.append({
            "doc_id": row["doc_id"],
            "section": row["section"],
            "text": row["text"],
            "semantic_score": float(semantic_scores[i]),
            "keyword_score": float(kw_score),
            "final_score": float(final_score)
        })
    results = sorted(results, key=lambda x: x["final_score"], reverse=True)
    return results[:top_k]

In [4]:
def rerank(query, retrieved_items):
    # Prosty reranking: premiujemy fragmenty zawierające definicje lub korzyści,
    # gdy pytanie brzmi jak pytanie wyjaśniające.
    query_lower = query.lower()
    reranked = []
    for item in retrieved_items:
        bonus = 0.0
        if "co to" in query_lower or "czym jest" in query_lower:
            if item["section"] in ["definition", "meaning"]:
                bonus += 0.10
        if "zalet" in query_lower or "korzy" in query_lower:
            if item["section"] == "benefits":
                bonus += 0.10
        if "ograniczen" in query_lower or "wady" in query_lower:
            if item["section"] == "limitations":
                bonus += 0.10

        item = item.copy()
        item["rerank_bonus"] = bonus
        item["reranked_score"] = item["final_score"] + bonus
        reranked.append(item)
    return sorted(reranked, key=lambda x: x["reranked_score"], reverse=True)

In [5]:
conversation_memory = []

def ask_rag(question, top_k=3):
    conversation_memory.append({"role": "user", "content": question})

    retrieved = hybrid_retrieve(question, top_k=5)
    reranked = rerank(question, retrieved)[:top_k]

    context_lines = []
    citations = []
    for i, item in enumerate(reranked, start=1):
        context_lines.append(f"[{i}] {item['text']}")
        citations.append(f"[{i}] {item['doc_id']}::{item['section']}")

    answer = (
        f"Pytanie: {question}\n\n"
        "Odpowiedź:\n"
        "Na podstawie odnalezionych źródeł system wskazuje, że RAG korzysta z zewnętrznego retrievalu, "
        "co pomaga ograniczać halucynacje i zwiększać kontrolę nad wiedzą. "
        "Jednocześnie skuteczność zależy od jakości danych, chunkingu i samego etapu retrieval.\n\n"
        "Najlepsze fragmenty:\n"
        + "\n".join(context_lines)
        + "\n\nCytowania:\n"
        + "\n".join(citations)
    )

    conversation_memory.append({"role": "assistant", "content": answer})
    return reranked, answer

In [6]:
question = "Czym jest RAG i jakie ma zalety?"
results, answer = ask_rag(question, top_k=3)

pd.DataFrame(results)

,doc_id,section,text,semantic_score,keyword_score,final_score,rerank_bonus,reranked_score
0,doc_rag,benefits,"Zaletą RAG jest redukcja halucynacji, większa ...",0.314578,0.428571,0.348776,0.1,0.448776
1,doc_rag,definition,"RAG to architektura, w której model generujący...",0.130613,0.142857,0.134286,0.1,0.234286
2,doc_rag,limitations,"Ograniczenia RAG obejmują słaby retrieval, błę...",0.117585,0.142857,0.125167,0.0,0.125167


In [7]:
print(answer)

Pytanie: Czym jest RAG i jakie ma zalety?

Odpowiedź:
Na podstawie odnalezionych źródeł system wskazuje, że RAG korzysta z zewnętrznego retrievalu, co pomaga ograniczać halucynacje i zwiększać kontrolę nad wiedzą. Jednocześnie skuteczność zależy od jakości danych, chunkingu i samego etapu retrieval.

Najlepsze fragmenty:
[1] Zaletą RAG jest redukcja halucynacji, większa kontrola nad źródłami i możliwość aktualizacji wiedzy bez pełnego trenowania modelu.
[2] RAG to architektura, w której model generujący korzysta z zewnętrznie pobranych fragmentów wiedzy.
[3] Ograniczenia RAG obejmują słaby retrieval, błędny chunking, niską jakość danych oraz problemy z rerankingiem.

Cytowania:
[1] doc_rag::benefits
[2] doc_rag::definition
[3] doc_rag::limitations


In [8]:
print("Pamięć konwersacji:")
for turn in conversation_memory:
    print(f"- {turn['role']}: {turn['content'][:120]}...")

Pamięć konwersacji:
- user: Czym jest RAG i jakie ma zalety?...
- assistant: Pytanie: Czym jest RAG i jakie ma zalety?

Odpowiedź:
Na podstawie odnalezionych źródeł system wskazuje, że RAG korzysta...


## Co ten notebook pokazuje
To już jest mini-architektura:
- retrieval semantyczny,
- retrieval słów kluczowych,
- ranking hybrydowy,
- reranking,
- pamięć sesji,
- źródła i cytowania.

Na zajęciach możesz pokazać, że **RAG nie jest jedną funkcją**, tylko całym pipeline'em.

## Rozszerzenia na dalsze zajęcia
Dalej można dołożyć:
- prawdziwe embeddingi,
- bazę wektorową,
- model LLM jako generator,
- ocenę jakości odpowiedzi,
- cache i filtrowanie metadanych.